# Owner and Transaction Scale Calculations
Process created by Nicholas Polimeni; Updated by Melissa Juarez to include more data cleaning

Code by Yuqi Zong

In [4]:
import pandas as pd
import os

pd.set_option('display.max_columns', 150)
pd.options.display.float_format = '{:,.3f}'.format

# set wd one folder back
os.chdir('/Users/melissajuarezc/Documents/GITHUB REPOS/parcel-data-processor/')


In [ ]:
# Cleaned digest and sales data from clean_data.ipynb
digest_full = pd.read_csv('/content/DIGEST_clayton_2012_2022.csv', low_memory=False)

digest_full['base_address'] = digest_full['streetaddress']
digest_full.loc[digest_full['base_address'].isna(), 'base_address'] = digest_full['address1']



In [3]:
digest_full[['streetaddress', 'address1', 'base_address']].sample(10)


,streetaddress,address1,base_address
590327,STE A,5360 JONESBORO ROAD,STE A
466978,NaN,3060 JODECO DR,3060 JODECO DR
280336,NaN,9485 FOREST KNOLL DR,9485 FOREST KNOLL DR
632750,NaN,8475 ALDEN COURT,8475 ALDEN COURT
885386,NaN,1987 LEVGARD LN,1987 LEVGARD LN
425224,STE B 525,5901 PEACHTREE DUNWOODY RD NE,STE B 525
32486,NaN,7126 HAZELWOOD DR,7126 HAZELWOOD DR
443758,NaN,324 BRITTAN TRAIL,324 BRITTAN TRAIL
320650,NaN,4615 BEAVERS RD,4615 BEAVERS RD
593311,NaN,PSC 78 BOX 2568,PSC 78 BOX 2568


In [4]:
#Clean address and extract unit numbers
digest_full['mod_own_adrstr'] = digest_full['base_address'].copy(deep=True)

digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(r'#(?!\s)', '# ', regex=True)

unit_pattern = r'(?:\b(?:unit|apt|ste|suite|suites|bldg)\b|#)[\s\.\-]*(.*)$'

digest_full['mod_unitno'] = digest_full['mod_own_adrstr'].str.extract(unit_pattern, flags=re.IGNORECASE)

digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(
    unit_pattern, '', flags=re.IGNORECASE, regex=True
).str.strip()

digest_full['mod_unitno'] = (
    digest_full['mod_unitno']
    .str.replace(r'^(?:unit|apt|ste|suite|suites|bldg)[\s\.\-]*', '', flags=re.IGNORECASE, regex=True)
    .str.replace(r'[#\-]', '', regex=True)
    .str.replace(r'\s+', '', regex=True)
    .fillna('')
)


In [5]:
digest_full[['base_address', 'mod_own_adrstr', 'mod_unitno']].sample(10)



,base_address,mod_own_adrstr,mod_unitno
636206,SUITE 425,,425
359709,130 TARRAGON DR,130 TARRAGON DR,
341246,6865 E SHERWOOD DR,6865 E SHERWOOD DR,
521806,730 NW 107TH AVENUE SUITE 400,730 NW 107TH AVENUE,400
984647,101 DEVANT ST STE 905,101 DEVANT ST,905
475617,11631 KADES TRAIL,11631 KADES TRAIL,
135928,3027 COTTON INDIAN CIR,3027 COTTON INDIAN CIR,
483050,725 EARHART ST,725 EARHART ST,
524095,5685 BUCKLEIGH PT,5685 BUCKLEIGH PT,
867925,1991 ROCK CUT PL,1991 ROCK CUT PL,


In [6]:
#Build a standardized owner address key
digest_full['owner_addr'] = (
    digest_full['mod_own_adrstr'].str.upper().fillna('') + '_' +
    digest_full['mod_unitno'].str.upper().fillna('') + '_' +
    digest_full['mailingcity'].str.upper().fillna('') + '_' +
    digest_full['mailingstate'].str.upper().fillna('') + '_' +
    digest_full['mailingzip'].str.upper().fillna('')
)
digest_full[['mod_own_adrstr', 'mod_unitno', 'mailingcity', 'mailingstate', 'mailingzip', 'owner_addr']].sample(10)


,mod_own_adrstr,mod_unitno,mailingcity,mailingstate,mailingzip,owner_addr
65194,241 S SALEM DRIVE,,MCDONOUGH,GA,30253,241 S SALEM DRIVE__MCDONOUGH_GA_30253
14149,1551 LAKE JODECO RD,,JONESBORO,GA,30236,1551 LAKE JODECO RD__JONESBORO_GA_30236
922526,8955 REDSKIN TRAIL,,JONESBORO,GA,30236,8955 REDSKIN TRAIL__JONESBORO_GA_30236
564355,,100,STOCKBRIDGE,GA,30281,_100_STOCKBRIDGE_GA_30281
453035,P O BOX 143741,,FAYETTEVILLE,GA,30214,P O BOX 143741__FAYETTEVILLE_GA_30214
406083,6439 KATIE LANE,,MORROW,GA,30260,6439 KATIE LANE__MORROW_GA_30260
583082,2769 TEAL LANDING DR,,MORROW,GA,30260,2769 TEAL LANDING DR__MORROW_GA_30260
947497,6372 WOODLAWN AVE,,REX,GA,30273,6372 WOODLAWN AVE__REX_GA_30273
58730,5138 WEST ST,,FOREST PARK,GA,30297,5138 WEST ST__FOREST PARK_GA_30297
142763,8025 WESTSIDE PARKWAY,,ALPHARETTA,GA,30009,8025 WESTSIDE PARKWAY__ALPHARETTA_GA_30009


## Identify corporate owners, create corp owner flags for each record
- grantee, grantor in sales
- own1 in digest

In [ ]:
#Flag corporate owners
corp_keywords = [
    'LLC', ' INC', 'LLP', 'L.L.C', 'L.L.P', 'I.N.C', 'L L C',
    'L L P', ' L P', ' LP', 'LTD', ' CORP', 'CORPORATION',
    'COMPANY', ' CO ', 'LIMITED', 'PARTNERSHIP', 'PARTNERSHIPS',
    'ASSOCIATION', 'ASSOC', 'INCORPORATED', 'INCORP',
    'L.T.D', 'LTD', "HOME", "SOLUTIONS"
]

digest_full['ownername1_upper'] = digest_full['taxpayername'].str.upper().fillna('')
digest_full['ownername2_upper'] = digest_full['additionalname'].str.upper().fillna('')

digest_full['own_corp_flag'] = digest_full.apply(
    lambda row: int(any(
        kw in row['ownername1_upper'] or kw in row['ownername2_upper']
        for kw in corp_keywords
    )),
    axis=1
)


In [9]:
digest_full[['taxpayername', 'additionalname', 'own_corp_flag']].sample(10)


,taxpayername,additionalname,own_corp_flag
298237,BRASWELL MORRIS E & MADELINE S,NaN,0
20291,WASHINGTON DAVID,NaN,0
738730,PHAM ANHTHU,NaN,0
616846,HOLLIS BOYD C OR DOROTHY C,NaN,0
181505,REAGIN WESLEY L OR PATRICIA L,NaN,0
636421,D R HORTON INC,NaN,1
418932,SEPANSKI THOMAS W,NaN,0
839874,ATL 3 SF LLC,% COLD RIVER LAND LLC,1
523828,WILLIAMS DAVID H,NaN,0
57259,MAJESTIC WATEROAK LLC,% NELKIN REAL ESTATE,1


In [10]:
# Count how many parcels each owner address holds per year
owner_yearly_holdings = (
    digest_full.groupby(['tax_year', 'owner_addr'])['pin']
    .count()
    .reset_index()
    .rename(columns={'pin': 'count_owned_clayton_yr'})
)

# Group all taxpayer names associated with an address
associated_owner_names = (
    digest_full.groupby('owner_addr')['taxpayername']
    .unique()
    .reset_index()
    .rename(columns={'taxpayername': 'assoc_owner_names'})
)

# Merge parcel counts with owner name lists
owner_scale = owner_yearly_holdings.merge(
    associated_owner_names,
    on='owner_addr',
    how='left'
)


In [11]:

owner_scale.sort_values(by='count_owned_clayton_yr', ascending=False).head(10)


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
746628,2022,"CENTURY PLAZA I,_350_ATLANTA_GA_30329",989,"[CLAYTON COUNTY LAND BANK, CLAYTON COUNTY LAND..."
615678,2021,112 SMITH STREET__JONESBORO_GA_30236,735,"[CLAYTON COUNTY GREENSPACE, CLAYTON COUNTY, CL..."
748484,2022,PO BOX 4090__SCOTTSDALE_AZ_85261,718,"[FREO GEORGIA LLC, PROGRESS RESIDENTIAL 2014-1..."
546845,2020,112 SMITH STREET__JONESBORO_GA_30236,642,"[CLAYTON COUNTY GREENSPACE, CLAYTON COUNTY, CL..."
155407,2014,21001 N TATUM BLVD_1630630_PHOENIX_AZ_85050,582,"[WILEY ALICISIA, THR GEORGIA LLC, WELLS FARGO ..."
34422,2012,4759 RYAN RD__CONLEY_GA_30288,562,"[ADAMS HOMES AEC LLC, DAVIS ODESSA MILLER, DAV..."
103042,2013,4759 RYAN RD__CONLEY_GA_30288,561,"[ADAMS HOMES AEC LLC, DAVIS ODESSA MILLER, DAV..."
170809,2014,4759 RYAN RD__CONLEY_GA_30288,561,"[ADAMS HOMES AEC LLC, DAVIS ODESSA MILLER, DAV..."
237867,2015,4759 RYAN RD__CONLEY_GA_30288,561,"[ADAMS HOMES AEC LLC, DAVIS ODESSA MILLER, DAV..."
304744,2016,4759 RYAN RD__CONLEY_GA_30288,521,"[ADAMS HOMES AEC LLC, DAVIS ODESSA MILLER, DAV..."


In [12]:
owner_scale[owner_scale['tax_year'] == 2022].sort_values(by='count_owned_clayton_yr', ascending=False).head(10)


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
746628,2022,"CENTURY PLAZA I,_350_ATLANTA_GA_30329",989,"[CLAYTON COUNTY LAND BANK, CLAYTON COUNTY LAND..."
748484,2022,PO BOX 4090__SCOTTSDALE_AZ_85261,718,"[FREO GEORGIA LLC, PROGRESS RESIDENTIAL 2014-1..."
749462,2022,_200_WEST LAKE HILLS_TX_78746,410,"[CPI/AMHERST SFR PROGRAM OWNER LLC, BAF ASSETS..."
716539,2022,5001 PLAZA ON THE LAKE_200_AUSTIN_TX_78746,392,"[BTRA V LLC, ARVM 5 LLC, BAF 1 LLC, SRMZ 1 LLC..."
749824,2022,_400_DULUTH_GA_30096,366,"[RHA 1 LLC, RHA 1 TRS LLC, RESI TL1 LLC, FYR S..."
709683,2022,3505 KOGER BLVD_400_DULUTH_GA_30096,353,"[ARLP REO 400 LLC, RESI SFR SUB LLC, HOME SFR ..."
749848,2022,_4100_ATLANTA_GA_30303,219,[CITY OF ATLANTA]
697126,2022,1850 PARKWAY PLACE_900_MARIETTA_GA_30067,218,"[ROSS HARIETT & JANSSEN DARLENE, BRADLEY ROBER..."
749433,2022,_200_AUSTIN_TX_78746,217,"[RPA4 LLC, HFS I ASSETS COMPANY LLC, JEFF 1 LL..."
685254,2022,112 SMITH STREET__JONESBORO_GA_30236,168,"[CLAYTON COUNTY GREENSPACE, CLAYTON COUNTY, CL..."


In [13]:
# Define institutional owner keywords
owner_keywords = {
    "Amherst": ["AMHERST", "ARVM"],
    "Cerberus": ["CERBERUS", "FKH", "RM1", "RMI"],
    "Progress": ["PROGRESS", "FREO"],
    "Invitation": ["INVITATION", "IH"],
    "Colony": ["COLONY", "STARWOOD", "CSH", "CAH"],
    "Sylvan": ["SYLVAN", "RNTR"],
    "Tricon": ["TRICON", "TAH"],
    "THR": ["THR", "WELLS", "WELLS FARGO"]
}

# Choose the year of interest
target_year = 2022

# Search and summarize
for label, keywords in owner_keywords.items():
    pattern = "|".join(keywords)

    filtered = owner_scale[
        (owner_scale["tax_year"] == target_year) &
        (owner_scale["assoc_owner_names"].apply(lambda names: any(
            re.search(pattern, name) for name in names if isinstance(name, str))
        ))
    ]

    filtered = filtered[filtered["count_owned_clayton_yr"] > 20]

    total_props = filtered["count_owned_clayton_yr"].sum()
    print(f"🏢 {label}: {total_props} properties in {target_year}")
    display(filtered.sort_values(by="count_owned_clayton_yr", ascending=False).head(5))


🏢 Amherst: 802 properties in 2022


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
749462,2022,_200_WEST LAKE HILLS_TX_78746,410,"[CPI/AMHERST SFR PROGRAM OWNER LLC, BAF ASSETS..."
716539,2022,5001 PLAZA ON THE LAKE_200_AUSTIN_TX_78746,392,"[BTRA V LLC, ARVM 5 LLC, BAF 1 LLC, SRMZ 1 LLC..."


🏢 Cerberus: 544 properties in 2022


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
697126,2022,1850 PARKWAY PLACE_900_MARIETTA_GA_30067,218,"[ROSS HARIETT & JANSSEN DARLENE, BRADLEY ROBER..."
750095,2022,_900_MARIETTA_GA_30067,157,"[CERBERUS SFR HOLDINGS II LP, CERBERUS SFR HOL..."
746539,2022,9TH FLR_900_MARIETTA_GA_30067,137,"[FKH SFR PROPCO D LP, RM1 SFR PROPCO A LP]"
746541,2022,9TH FLR__MARIETTA_GA_30067,32,[RM1 SFR PROPCO A LP]


🏢 Progress: 783 properties in 2022


...

,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
749230,2022,_11STE300_ATLANTA_GA_30305,117,"[RNTR-3 LLC, RNTR-1 LLC, RNTR-2 LLC, VSP ATLAN..."


🏢 Tricon: 278 properties in 2022


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
693016,2022,1508 BROOKHOLLOW DR__SANTA ANA_CA_92705,128,"[2015B PROPERTY OWNER LLC, 2016A PROPERTY OWNE..."
693014,2022,1508 BROOKHOLLOW DRIVE__SANTA ANA_CA_92705,93,"[TAH 2017-2 BORROWER LLC, SFR JV-1 2020-1 BORR..."
748138,2022,PO BOX 15087__SANTA ANA_CA_92735,57,"[THORPE JOHN WALTER OR, CHADWICK BENJAMIN L OR..."


🏢 THR: 286 properties in 2022


,tax_year,owner_addr,count_owned_clayton_yr,assoc_owner_names
749433,2022,_200_AUSTIN_TX_78746,217,"[RPA4 LLC, HFS I ASSETS COMPANY LLC, JEFF 1 LL..."
749796,2022,_390_SCOTTSDALE_AZ_85251,39,"[SPH PROPERTY TWO LLC, SPH PROPERTY ONE LLC, S..."
749416,2022,_2000_DALLAS_TX_75201,30,"[THR GEORGIA LP, 2017-1 IH BORROWER LP, THE GE..."


In [14]:
# Export full cleaned dataset
digest_full.to_csv('/content/clayton_cleaned_full.csv', index=False)

# Export owner-scale aggregation
owner_scale.to_csv('/content/clayton_owner_scale.csv', index=False)




In [15]:
# Re-load the exported CSV just to confirm structure
df_check = pd.read_csv('/content/clayton_cleaned_full.csv')

# Show key columns
df_check[['pin', 'tax_year', 'taxpayername', 'address1',
          'mod_own_adrstr', 'mod_unitno', 'owner_addr', 'own_corp_flag']].sample(10)


/tmp/ipython-input-249762565.py:2: DtypeWarning: Columns (3,11,19,20,21,23,28,40,45,68,72,77,78,83,86,89,92,95,98,101,104,107,122,123,125,127,131,133,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv('/content/clayton_cleaned_full.csv')


,pin,tax_year,taxpayername,address1,mod_own_adrstr,mod_unitno,owner_addr,own_corp_flag
434439,13149C G008,2016,BELL HELEN P,232 SHENANDOAH DR,232 SHENANDOAH DR,NaN,232 SHENANDOAH DR__RIVERDALE_GA_30274,0
790348,13111A D038,2020,NGUYEN VU MINH,633 BROOKWOOD DRIVE,633 BROOKWOOD DRIVE,NaN,633 BROOKWOOD DRIVE__FOREST PARK_GA_30297,0
364391,05079C D015,2016,HERITAGE PROPERTY HOLDINGS LLC,1266 W PACES FERRY RD #586,1266 W PACES FERRY RD,586,1266 W PACES FERRY RD_586_ATLANTA_GA_30327,1
280600,05214D H008,2015,AJUEYITSI DAVID O,5020 GUILFORD FOREST DR,5020 GUILFORD FOREST DR,NaN,5020 GUILFORD FOREST DR__ATLANTA_GA_30331,0
914336,05177C A010,2022,BARBER TASHINA S OR DARRELL,10168 POINT VIEW DR,10168 POINT VIEW DR,NaN,10168 POINT VIEW DR__JONESBORO_GA_30238,0
648662,06028C B011,2019,MOORE JERMAINE S,9223 FAIRFIELD APPROACH,9223 FAIRFIELD APPROACH,NaN,9223 FAIRFIELD APPROACH__JONESBORO_GA_30236,0
886027,13137A B001,2021,QUINTERO JUAN,4012 PANOLA RD,4012 PANOLA RD,NaN,4012 PANOLA RD__LITHONIA_GA_30038,0
569201,12018A A026,2018,BLACK NANCY JEAN,8192 SUNNYDALE LANE,8192 SUNNYDALE LANE,NaN,8192 SUNNYDALE LANE__JONESBORO_GA_30236,0
992726,13212C B015,2022,CORREA FERNANDO MARTINEZ,58 SKYLARK LANE,58 SKYLARK LANE,NaN,58 SKYLARK LANE__JONESBORO_GA_30238,1
575688,12087D D013,2018,WIGGINS CAROLYN E,7142 HAZELWOOD DR,7142 HAZELWOOD DR,NaN,7142 HAZELWOOD DR__JONESBORO_GA_30236,0
